# Phase 9: Graph Neural Networks (LightGCN)

In the previous phase, we discovered an "Embedding Gap": simple heuristics (Jaccard) outperformed our random-walk embeddings (Node2Vec). This is because random walks are generic; they don't explicitly optimize for the *collaborative filtering* signal.

**LightGCN** (Lightweight Graph Convolutional Network) is designed to solve this. It is the modern "Gold Standard" for link prediction in bipartite graphs.

## 1. Why LightGCN?
- **Simplicity:** It removes complex non-linearities (like ReLU) and heavyweight weight matrices. 
- **Message Passing:** It learns node embeddings by explicitly averaging the features of a node's neighbors across multiple layers. 
- **Efficiency:** It is highly optimized for large-scale recommendation tasks.

## 2. The Plan (Step-by-Step)
1. **Data Transformation:** Convert our NetworkX graph into PyTorch Geometric `HeteroData` tensors.
2. **The Model:** Implement the LightGCN architecture with BPR (Bayesian Personalized Ranking) loss.
3. **Training:** Optimize embeddings using positive (seen) and negative (unseen) interactions.
4. **Evaluation:** Integrate LightGCN into our Unified Benchmark to see if it beats Jaccard.

## Step 1: Data Preparation and Node Mapping

PyTorch Geometric works with integer indices, not strings. Our first task is to create a mapping from our `u_1`, `m_1` string IDs to sequential integers `0, 1, 2...`.

### Why separate indices?
Since this is a **Bipartite Graph**, we maintain two separate index systems:
- **Users:** 0 to 942 (Total 943)
- **Movies:** 0 to 1681 (Total 1682)

This allows the model to differentiate between a user node and a movie node during the message-passing phase.

In [1]:
import torch
import pandas as pd
import numpy as np
from torch_geometric.data import HeteroData
from sklearn.model_selection import train_test_split

# 1. Load Data
r_cols = ['user_id', 'movie_id', 'rating', 'ts']
df = pd.read_csv('data/u.data', sep='\t', names=r_cols)

# 2. Filter for Positive Links (Ratings >= 4)
pos_df = df[df['rating'] >= 4].copy()

# 3. Create Continuous ID Mappings
# LightGCN needs 0-indexed contiguous integers for embedding lookups.
user_map = {old: i for i, old in enumerate(sorted(df['user_id'].unique()))}
movie_map = {old: i for i, old in enumerate(sorted(df['movie_id'].unique()))}

num_users = len(user_map)
num_movies = len(movie_map)

# 4. Apply mapping to our positive edges
pos_df['u_idx'] = pos_df['user_id'].map(user_map)
pos_df['m_idx'] = pos_df['movie_id'].map(movie_map)

# 5. Train/Test Split (80/20)
train_edges, test_edges = train_test_split(pos_df, test_size=0.2, random_state=42)

# 6. Convert to PyTorch Tensors
# The 'edge_index' is a 2xN tensor where row 0 is users and row 1 is movies.
edge_index = torch.stack([
    torch.tensor(train_edges['u_idx'].values, dtype=torch.long),
    torch.tensor(train_edges['m_idx'].values, dtype=torch.long)
])

print(f"Nodes: {num_users} Users, {num_movies} Movies")
print(f"Training Edges: {edge_index.shape[1]}")
print(f"Edge Index Shape: {edge_index.shape}")

c:\Users\RoG\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Nodes: 943 Users, 1682 Movies
Training Edges: 44300
Edge Index Shape: torch.Size([2, 44300])


## Step 2: Defining the LightGCN Architecture

We will implement the model using `torch.nn.Module`. The core logic follows the "Linear Message Passing" rule: 
Nodes propagate their features to neighbors without any non-linear activations (like ReLU) or weight matrices. This simplicity is why it's called "Light" GCN.

### Why Layer Combination?
- **Layer 0:** Captures the node's individual characteristics.
- **Layer 1:** Captures immediate neighbors (Collaborative Filtering).
- **Layer 2 & 3:** Captures "friends of friends" (Higher-order structural similarity).

By averaging all layers, the model learns a balanced representation.

In [2]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import LGConv

class LightGCN(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=64, num_layers=3):
        super(LightGCN, self).__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.embedding_dim = embedding_dim
        self.num_layers = num_layers

        # 1. Initialize Embeddings
        # These are the 'Layer 0' embeddings that will be optimized during training.
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)

        # Initialize with small random values for better convergence
        nn.init.normal_(self.user_embedding.weight, std=0.1)
        nn.init.normal_(self.item_embedding.weight, std=0.1)

        # 2. Initialize the Convolution Layer (LightGCN specific)
        self.convs = nn.ModuleList([LGConv() for _ in range(num_layers)])

    def get_initial_embeddings(self):
        # Combine user and item embeddings into one matrix for the graph convolution
        return torch.cat([self.user_embedding.weight, self.item_embedding.weight], dim=0)

    def forward(self, edge_index):
        """
        The forward pass propagates embeddings across the graph.
        """
        # We need an undirected graph where users connect to items AND vice-versa.
        # PyG LGConv expects a symmetric edge_index.
        user_indices, item_indices = edge_index[0], edge_index[1]
        
        # Shift item indices so they don't overlap with user indices (e.g., item 0 becomes 943)
        adj_edge_index = torch.stack([
            torch.cat([user_indices, item_indices + self.num_users]),
            torch.cat([item_indices + self.num_users, user_indices])
        ], dim=0)

        x = self.get_initial_embeddings()
        all_layers = [x] # Store embeddings from every layer

        # Perform Message Passing across num_layers
        for conv in self.convs:
            x = conv(x, adj_edge_index)
            all_layers.append(x)

        # Final embedding is the average of all layer embeddings
        final_embeddings = torch.mean(torch.stack(all_layers, dim=0), dim=0)
        
        # Split back into users and items
        users_emb, items_emb = torch.split(final_embeddings, [self.num_users, self.num_items])
        return users_emb, items_emb

model = LightGCN(num_users, num_movies)
print(model)

LightGCN(
  (user_embedding): Embedding(943, 64)
  (item_embedding): Embedding(1682, 64)
  (convs): ModuleList(
    (0-2): 3 x LGConv()
  )
)


## Step 3: Training with BPR Loss

We use **Bayesian Personalized Ranking (BPR)** loss. Unlike standard binary cross-entropy, BPR is a **ranking loss**. It forces the model to rank items the user liked higher than items they haven't seen.

### Why Negative Sampling?
Our dataset only contains "Positive Links" (movies users liked). Without negative sampling, the model would simply learn to give every pair a high score. By sampling "Negative Links" (movies the user hasn't seen), we teach the model the difference between a high-probability link and a low-probability one.

In [3]:
torch.device('cuda' if torch.cuda.is_available() else 'cpu')

device(type='cpu')

In [4]:
import random
from torch.optim import Adam

def get_bpr_loss(users_emb, pos_items_emb, neg_items_emb, lambda_reg=1e-4):
    """
    Calculates BPR Loss: -sum(log(sigmoid(pos_score - neg_score))) + regularization
    """
    pos_scores = torch.mul(users_emb, pos_items_emb).sum(dim=1)
    neg_scores = torch.mul(users_emb, neg_items_emb).sum(dim=1)

    loss = -torch.log(torch.sigmoid(pos_scores - neg_scores)).mean()
    
    # L2 Regularization helps prevent the embeddings from growing too large (overfitting)
    reg_loss = lambda_reg * (users_emb.norm(2).pow(2) + 
                           pos_items_emb.norm(2).pow(2) + 
                           neg_items_emb.norm(2).pow(2)) / users_emb.size(0)
    
    return loss + reg_loss

def sample_negative_items(edge_index, num_items):
    """
    For each user in the edge_index, find a random movie they haven't interacted with.
    """
    users, pos_items = edge_index[0], edge_index[1]
    neg_items = []
    for u in users:
        while True:
            neg_i = random.randint(0, num_items - 1)
            # Basic check: ideally we verify this user hasn't seen this movie in G_train
            if neg_i not in pos_items[users == u]:
                neg_items.append(neg_i)
                break
    return torch.tensor(neg_items, dtype=torch.long)

# --- Start Training ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
edge_index = edge_index.to(device)
optimizer = Adam(model.parameters(), lr=0.01)

print(f"Training on {device}...")
for epoch in range(1, 101):
    model.train()
    optimizer.zero_grad()
    
    # 1. Forward Pass: Get final embeddings
    u_emb, i_emb = model(edge_index)
    
    # 2. Negative Sampling
    neg_indices = sample_negative_items(edge_index, num_movies).to(device)
    
    # 3. Get Embeddings for the sampled triplets
    # We use the original user and positive item indices from our training set
    train_u_idx, train_i_idx = edge_index[0], edge_index[1]
    
    # 4. Calculate Loss
    loss = get_bpr_loss(u_emb[train_u_idx], i_emb[train_i_idx], i_emb[neg_indices])
    
    # 5. Backprop
    loss.backward()
    optimizer.step()
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch:03d} | Loss: {loss.item():.4f}")
#8mn54 on cpu

Training on cpu...
Epoch 010 | Loss: 0.5577
Epoch 020 | Loss: 0.3266
Epoch 030 | Loss: 0.2996
Epoch 040 | Loss: 0.2907
Epoch 050 | Loss: 0.2738
Epoch 060 | Loss: 0.2636
Epoch 070 | Loss: 0.2430
Epoch 080 | Loss: 0.2281
Epoch 090 | Loss: 0.2158
Epoch 100 | Loss: 0.2025


## Step 4: Evaluation & Integration

Now that we have trained our GNN, we need to see how it performs compared to our Jaccard baseline. 

### The Prediction Logic
In LightGCN, a user's preference for a movie is simply the **Dot Product** of their final embeddings. To recommend items, we:
1. Get the target user's final vector.
2. Calculate its similarity with every movie's vector.
3. Rank them and return the top 10.

In [5]:
def recommend_lightgcn(target_user_node, model, edge_index, user_map, movie_map, k=10):
    """
    Generates Top-K recommendations for a user node (e.g., 'u_1') using the trained GNN.
    """
    model.eval()
    with torch.no_grad():
        # 1. Get all final embeddings
        u_emb, i_emb = model(edge_index)
        
        # 2. Get the specific user index
        u_id = int(target_user_node.replace('u_', ''))
        u_idx = user_map[u_id]
        
        # 3. Calculate scores for all movies (Dot Product)
        user_vector = u_emb[u_idx]
        scores = torch.matmul(user_vector, i_emb.t())
        
        # 4. Filter out movies the user has already seen
        seen_movies = edge_index[1][edge_index[0] == u_idx].tolist()
        scores[seen_movies] = -np.inf # Give seen movies a very low score
        
        # 5. Get Top-K movie indices
        top_k_indices = torch.topk(scores, k).indices.cpu().numpy()
        
        # 6. Map back to 'm_ID' format
        rev_movie_map = {v: k for k, v in movie_map.items()}
        return ['m_' + str(rev_movie_map[idx]) for idx in top_k_indices]

# --- Example Recommendation ---
target = 'u_1'
recs = recommend_lightgcn(target, model, edge_index, user_map, movie_map)
print(f"Top 10 LightGCN Recs for {target}:\n", recs)

# --- Final Performance Check (on same metrics) ---
def get_metrics(recs, hidden_set):
    hits = set(recs).intersection(hidden_set)
    return len(hits) / len(recs) if recs else 0

precisions = []
test_users = test_edges['u_idx'].unique()[:100] # Test on 100 users for speed
for u_idx in test_users:
    u_node = 'u_' + str(list(user_map.keys())[list(user_map.values()).index(u_idx)])
    hidden = set('m_' + str(list(movie_map.keys())[list(movie_map.values()).index(m)]) 
                 for m in test_edges[test_edges['u_idx'] == u_idx]['m_idx'])
    
    recs = recommend_lightgcn(u_node, model, edge_index, user_map, movie_map)
    precisions.append(get_metrics(recs, hidden))

print(f"\nLightGCN Mean Precision@10: {np.mean(precisions):.4f}")

Top 10 LightGCN Recs for u_1:
 ['m_100', 'm_127', 'm_318', 'm_357', 'm_483', 'm_168', 'm_197', 'm_12', 'm_69', 'm_427']

LightGCN Mean Precision@10: 0.2970


## Final Interpretation: Did we beat Jaccard?

By implementing **LightGCN**, we moved from simple structure-matching to **explicit feature propagation**. 

### Why this works better than Node2Vec:
While Node2Vec treats users and movies as generic nodes in a walk, LightGCN explicitly optimizes the embeddings to satisfy the **Ranking Objective (BPR)**. It essentially learns a "Collaborative Filtering Space" where proximity directly correlates with the probability of a user liking a movie.

### Future Improvements:
- **Data Augmentation:** We can use "Self-Supervised Learning" (like Contrastive Learning) to make the embeddings even more robust to noise.
- **Feature Injection:** We can initialize $E^{(0)}$ with the User/Movie attributes from Phase 5 instead of random noise.